In [ ]:
# ============================================================
# GDP DIRECTION CLASSIFIER
# Predict whether GDP growth improves next year
# ============================================================

import os
import joblib
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
os.makedirs("models", exist_ok=True)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import cross_validate
from sklearn.base import clone

import xgboost as xgb


# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------

df = pd.read_csv("data/03_panel_instability.csv")

df = df.sort_values(["COUNTRY", "YEAR"]).copy()

print(f"Loaded: {df.shape}")
print(f"Countries: {df['COUNTRY'].nunique()}")
print(f"Years: {df['YEAR'].min()}-{df['YEAR'].max()}")


# ------------------------------------------------------------
# Binary target
# 1 = GDP growth improved compared with previous year
# 0 = GDP growth stayed same or declined
# ------------------------------------------------------------

df["GDP_Up"] = (
    df["GDP_Growth"] > df["GDP_Growth_lag1"]
).astype(int)

print(df["GDP_Up"].value_counts(normalize=True).round(3))


# ------------------------------------------------------------
# Feature columns
# Use only lagged / historical features
# ------------------------------------------------------------

feature_cols = [
    "GDP_Growth_lag1",
    "GDP_Growth_rollmean3",
    "Inflation_lag1_log",
    "Exports_lag1",
    "Imports_lag1",
    "Fiscal_Balance_lag1",
    "Current_Account_lag1",
    "Debt_diff_lag1",
    "Expenditure_diff_lag1",
    "Revenue_diff_lag1",
    "Savings_diff_lag1",
    "Investment_diff_lag1",
    "Instability_Index_lag1",
]

# Add volatility features if available
volatility_cols = [
    col for col in [
        "GDP_Growth_rollstd3",
        "Inflation_rollstd3",
        "Exports_rollstd3",
        "Imports_rollstd3",
        "Fiscal_Balance_rollstd3",
        "Current_Account_rollstd3",
        "Debt_rollstd3",
    ]
    if col in df.columns
]

feature_cols = feature_cols + volatility_cols

df_model = df.dropna(
    subset=feature_cols + ["GDP_Up", "GDP_Growth"]
).copy()

print(f"Model dataset: {df_model.shape}")
print(f"Features used: {len(feature_cols)}")


# ------------------------------------------------------------
# Train / test / forecast split
# Main evaluation excludes COVID years 2020-2021
# ------------------------------------------------------------

TRAIN_END = 2019
TEST_START = 2022
TEST_END = 2023

train = df_model[df_model["YEAR"] <= TRAIN_END].copy()

test = df_model[
    (df_model["YEAR"] >= TEST_START)
    & (df_model["YEAR"] <= TEST_END)
].copy()

forecast = df_model[
    df_model["YEAR"].isin([2024, 2025, 2026])
].copy()

X_train = train[feature_cols].to_numpy()
y_train = train["GDP_Up"].to_numpy()

X_test = test[feature_cols].to_numpy()
y_test = test["GDP_Up"].to_numpy()

X_forecast = forecast[feature_cols].to_numpy()

print(f"Train: {train['YEAR'].min()}-{train['YEAR'].max()} | n={len(train)}")
print(f"Test : {test['YEAR'].min()}-{test['YEAR'].max()} | n={len(test)}")
print(f"Forecast rows 2024-2026: {len(forecast)}")


# ------------------------------------------------------------
# Expanding-window CV
# ------------------------------------------------------------

train_years = train["YEAR"].to_numpy(copy=False)

validation_years = [2015, 2016, 2017, 2018, 2019]

time_splits = []

for validation_year in validation_years:
    train_idx = np.flatnonzero(train_years < validation_year)
    validation_idx = np.flatnonzero(train_years == validation_year)

    if len(train_idx) > 0 and len(validation_idx) > 0:
        time_splits.append((train_idx, validation_idx))

        print(
            f"Fold {validation_year}: "
            f"train={len(train_idx)}, validation={len(validation_idx)}"
        )


# ------------------------------------------------------------
# Classifier models
# ------------------------------------------------------------

positive_rate = y_train.mean()
negative_rate = 1 - positive_rate

scale_pos_weight = (
    negative_rate / positive_rate
    if positive_rate > 0
    else 1
)

classifiers = {
    "Logistic Regression": {
        "model": LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            random_state=42,
        ),
        "scaled": True,
    },

    "Random Forest": {
        "model": RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=1,
        ),
        "scaled": False,
    },

    "XGBoost": {
        "model": xgb.XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            scale_pos_weight=scale_pos_weight,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            verbosity=0,
            n_jobs=1,
        ),
        "scaled": False,
    },
}


# ------------------------------------------------------------
# Cross-validation + final training
# ------------------------------------------------------------

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

trained_classifiers = {}
cv_rows = []

for name, cfg in classifiers.items():
    print(f"\nTraining {name}...")

    estimator = clone(cfg["model"])

    if cfg["scaled"]:
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", estimator),
        ])
    else:
        pipeline = Pipeline([
            ("model", estimator),
        ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=time_splits,
        scoring=scoring,
        n_jobs=1,
        error_score="raise",
    )

    row = {
        "Model": name,
        "CV_Accuracy": scores["test_accuracy"].mean(),
        "CV_Precision": scores["test_precision"].mean(),
        "CV_Recall": scores["test_recall"].mean(),
        "CV_F1": scores["test_f1"].mean(),
        "CV_ROC_AUC": scores["test_roc_auc"].mean(),
    }

    cv_rows.append(row)

    pipeline.fit(X_train, y_train)

    trained_classifiers[name] = pipeline

    joblib.dump(
        pipeline,
        f"models/gdp_direction_{name.lower().replace(' ', '_')}.pkl"
    )

    print(
        f"{name:<20} "
        f"Accuracy={row['CV_Accuracy']:.3f} | "
        f"F1={row['CV_F1']:.3f} | "
        f"ROC-AUC={row['CV_ROC_AUC']:.3f}"
    )

cv_results_df = pd.DataFrame(cv_rows)
cv_results_df.to_csv(
    "data/gdp_direction_cv_results.csv",
    index=False,
)

display(cv_results_df.sort_values("CV_ROC_AUC", ascending=False))


# ------------------------------------------------------------
# Test evaluation: observed 2022-2023 only
# ------------------------------------------------------------

test_rows = []

for name, pipeline in trained_classifiers.items():
    y_pred = pipeline.predict(X_test)

    if hasattr(pipeline, "predict_proba"):
        y_prob = pipeline.predict_proba(X_test)[:, 1]
    else:
        y_prob = y_pred

    row = {
        "Model": name,
        "Period": "Observed test (2022-23)",
        "N": len(y_test),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_prob),
    }

    test_rows.append(row)

    print(f"\n{name}")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, zero_division=0))

test_results_df = pd.DataFrame(test_rows)

test_results_df.to_csv(
    "data/gdp_direction_test_results.csv",
    index=False,
)

display(test_results_df.sort_values("ROC_AUC", ascending=False))


# ------------------------------------------------------------
# Forecast probabilities for 2024-2026
# These are IMF projection-period scenario outputs, not actual accuracy
# ------------------------------------------------------------

forecast_output = forecast[["COUNTRY", "YEAR"]].reset_index(drop=True)

for name, pipeline in trained_classifiers.items():
    pred_class = pipeline.predict(X_forecast)
    pred_prob = pipeline.predict_proba(X_forecast)[:, 1]

    safe_name = name.replace(" ", "_")

    forecast_output[f"{safe_name}_GDP_Up_Class"] = pred_class
    forecast_output[f"{safe_name}_GDP_Up_Prob"] = pred_prob

forecast_output.to_csv(
    "data/gdp_direction_forecasts_2024_2026.csv",
    index=False,
)

print("Saved:")
print("  data/gdp_direction_cv_results.csv")
print("  data/gdp_direction_test_results.csv")
print("  data/gdp_direction_forecasts_2024_2026.csv")